In [ ]:
!pip install tensorflow

In [ ]:
# --------------------
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense
from sklearn.metrics import matthews_corrcoef, cohen_kappa_score
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import GRU, Dropout, Dense, Input, Reshape, MaxPooling1D
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, roc_auc_score
from keras.layers import Dense, Conv1D, MaxPool1D, Flatten, Dropout,LSTM
from sklearn.model_selection import KFold

# Enable eager execution
tf.config.run_functions_eagerly(True)
tf.config.experimental_run_functions_eagerly(True)

In [ ]:
!pip install catboost

In [ ]:
from sklearn import metrics
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, cohen_kappa_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import matthews_corrcoef
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.ensemble import AdaBoostClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import StackingClassifier
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score, cross_val_predict

In [ ]:
# Load your data
train = pd.read_csv('/content/LSA-Tr.csv')

xtrain = train.drop(['Target'], axis=1)
ytrain = train['Target']

In [ ]:
from imblearn.over_sampling import ADASYN
ada = ADASYN()
xtrain, ytrain = ada.fit_resample(xtrain, ytrain)

In [ ]:
def build_stacking():
    estimators = [
        ('RF', RandomForestClassifier(n_estimators=450, max_depth=9, random_state=42)),
        ('XGB', XGBClassifier(n_estimators=350, max_depth=7, learning_rate=0.1,
                              eval_metric='logloss', use_label_encoder=False)),
        ('Cat', CatBoostClassifier(depth=7, iterations=45, learning_rate=0.35, verbose=0)),
        ('LGBM', LGBMClassifier(learning_rate=0.1, max_depth=7, random_state=50)),
        ('ETC', ExtraTreesClassifier(n_estimators=450, max_depth=7, random_state=42)),
        ('KNN', KNeighborsClassifier(n_neighbors=5)),
        ('ADB', AdaBoostClassifier(n_estimators=350, learning_rate=0.5, random_state=50))
    ]

    meta = XGBClassifier(
        n_estimators=350, max_depth=7,
        learning_rate=0.1, eval_metric='logloss',
        use_label_encoder=False
    )

    return StackingClassifier(
        estimators=estimators,
        final_estimator=meta,
        passthrough=False,
        n_jobs=-1
    )

In [ ]:
def federated_stacking_training(clients, X_val, rounds=25, trim_ratio=0.2):
    client_models = []

    # ---- Local training
    for i, client in enumerate(clients):
        model = build_stacking()
        model.fit(client['x'].reshape(client['x'].shape[0], -1), client['y'])
        client_models.append(model)
        print(f"Client {i+1} trained")

    # ---- Federated prediction aggregation
    all_probs = []

    for model in client_models:
        probs = model.predict_proba(
            X_val.reshape(X_val.shape[0], -1)
        )[:, 1]
        all_probs.append(probs)

    all_probs = np.array(all_probs)

    # ---- Trimmed Mean aggregation
    sorted_probs = np.sort(all_probs, axis=0)
    k = int(trim_ratio * len(client_models))
    trimmed = sorted_probs[k:len(client_models)-k]
    global_probs = np.mean(trimmed, axis=0)

    return global_probs

In [ ]:
NUM_CLIENTS = 5

# Convert to NumPy (2D)
X_train_tab = xtrain.to_numpy()
Y_train_tab = ytrain.to_numpy()

# Separate positive & negative samples
pos_idx = np.where(Y_train_tab == 1)[0]
neg_idx = np.where(Y_train_tab == 0)[0]

# Shuffle
np.random.shuffle(pos_idx)
np.random.shuffle(neg_idx)

# Split equally
pos_split = np.array_split(pos_idx, NUM_CLIENTS)
neg_split = np.array_split(neg_idx, NUM_CLIENTS)

clients = []
for i in range(NUM_CLIENTS):
    idx = np.concatenate([pos_split[i], neg_split[i]])
    np.random.shuffle(idx)

    clients.append({
        'x': X_train_tab[idx],   # 2D
        'y': Y_train_tab[idx]
    })

# Verify distribution
for i, c in enumerate(clients):
    print(f"Client {i+1}: total={len(c['y'])}, pos={np.sum(c['y'])}, neg={len(c['y'])-np.sum(c['y'])}")

Client 1: total=900, pos=450, neg=450
Client 2: total=900, pos=450, neg=450
Client 3: total=900, pos=450, neg=450
Client 4: total=900, pos=450, neg=450
Client 5: total=900, pos=450, neg=450


In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=1)
train_idx, val_idx = next(kf.split(X_train_tab))

X_val_fold = X_train_tab[val_idx]
Y_val_fold = Y_train_tab[val_idx]

In [ ]:
global_probs = federated_stacking_training(clients, X_val_fold, rounds=25)

preds = (global_probs > 0.5).astype(int)

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:27:10] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 1 trained


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:28:13] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 2 trained


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:29:15] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


...

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:30:17] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 4 trained


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:31:24] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 5 trained


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
from sklearn.metrics import (confusion_matrix, accuracy_score, recall_score,precision_score, f1_score, matthews_corrcoef,
                             cohen_kappa_score)

cm = confusion_matrix(Y_val_fold, preds)
specificity = cm[0,0]/(cm[0,0]+cm[0,1])

print("\nConfusion Matrix:\n", cm)
print("Accuracy :", accuracy_score(Y_val_fold, preds))
print("Sensitivity (Recall):", recall_score(Y_val_fold, preds))
print("Specificity:", specificity)
print("MCC      :", matthews_corrcoef(Y_val_fold, preds))
print("F1       :", f1_score(Y_val_fold, preds))
print("Precision:", precision_score(Y_val_fold, preds))
print("Kappa    :", cohen_kappa_score(Y_val_fold, preds))


Confusion Matrix:
 [[439   1]
 [  1 459]]
Accuracy : 0.9977777777777778
Sensitivity (Recall): 0.9978260869565218
Specificity: 0.9977272727272727
MCC      : 0.9955533596837944
F1       : 0.9978260869565218
Precision: 0.9978260869565218
Kappa    : 0.9955533596837944


**Testing**

In [ ]:
# Load your data
test = pd.read_csv('/content/LSA-Ind.csv')

xtest = test.drop(['Target'], axis=1)
ytest = test['Target']

In [ ]:
NUM_CLIENTS = 5

# Convert to NumPy (2D)
X_train_tab = xtrain.to_numpy()
Y_train_tab = ytrain.to_numpy()

# Separate positive & negative samples
pos_idx = np.where(Y_train_tab == 1)[0]
neg_idx = np.where(Y_train_tab == 0)[0]

# Shuffle
np.random.shuffle(pos_idx)
np.random.shuffle(neg_idx)

# Split equally
pos_split = np.array_split(pos_idx, NUM_CLIENTS)
neg_split = np.array_split(neg_idx, NUM_CLIENTS)

clients = []
for i in range(NUM_CLIENTS):
    idx = np.concatenate([pos_split[i], neg_split[i]])
    np.random.shuffle(idx)

    clients.append({
        'x': X_train_tab[idx],   # 2D
        'y': Y_train_tab[idx]
    })

# Verify distribution
for i, c in enumerate(clients):
    print(f"Client {i+1}: total={len(c['y'])}, pos={np.sum(c['y'])}, neg={len(c['y'])-np.sum(c['y'])}")

Client 1: total=900, pos=450, neg=450
Client 2: total=900, pos=450, neg=450
Client 3: total=900, pos=450, neg=450
Client 4: total=900, pos=450, neg=450
Client 5: total=900, pos=450, neg=450


In [ ]:
client_models = []

for i, client in enumerate(clients):
    model = build_stacking()
    model.fit(client['x'], client['y'])
    client_models.append(model)
    print(f"Client {i+1} trained")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:32:29] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 1 trained


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:33:29] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 2 trained


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:34:31] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 3 trained


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:35:31] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Client 4 trained
Client 5 trained


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:199: UserWarning: [16:36:35] WARNING: /workspace/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
X_test_tab = xtest.to_numpy()
Y_test_tab = ytest.to_numpy()

all_probs = []

for model in client_models:
    probs = model.predict_proba(X_test_tab)[:, 1]
    all_probs.append(probs)

all_probs = np.array(all_probs)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [ ]:
trim_ratio = 0.2
k = int(trim_ratio * len(client_models))

sorted_probs = np.sort(all_probs, axis=0)
trimmed = sorted_probs[k:len(client_models)-k]

global_probs = np.mean(trimmed, axis=0)

preds = (global_probs > 0.5).astype(int)

In [ ]:
cm = confusion_matrix(Y_test_tab, preds)
specificity = cm[0,0]/(cm[0,0]+cm[0,1])

print("\nConfusion Matrix:\n", cm)
print("Accuracy :", accuracy_score(Y_test_tab, preds))
print("Sensitivity (Recall):", recall_score(Y_test_tab, preds))
print("Specificity:", specificity)
print("MCC      :", matthews_corrcoef(Y_test_tab, preds))
print("F1       :", f1_score(Y_test_tab, preds))
print("Precision:", precision_score(Y_test_tab, preds))
print("Kappa    :", cohen_kappa_score(Y_test_tab, preds))


Confusion Matrix:
 [[553   9]
 [ 46  10]]
Accuracy : 0.9110032362459547
Sensitivity (Recall): 0.17857142857142858
Specificity: 0.9839857651245552
MCC      : 0.27031998386715983
F1       : 0.26666666666666666
Precision: 0.5263157894736842
Kappa    : 0.23137804712586496
